# Validation du regex d'extraction d'articles

**Objectif.** Mesurer la fiabilité du regex `extract_code_article_pairs()` défini dans `enrichissement_base_complete.ipynb` qui a produit les champs `code_article_pairs` de toute la base enrichie (1,1 M arrêts).

**Protocole.**
1. Échantillonner **100 arrêts** stratifiés : 50 CC + 25 CA + 25 TJ depuis `database-judilibre-enrichie/`.
2. **Baseline regex** : lire directement le champ `code_article_pairs` déjà calculé par le regex dans la base enrichie (c'est exactement l'output en production).
3. **LLM** : ré-extraire les articles depuis le `text` via vLLM local (Gemma 4 par défaut) en JSON strict.
4. **Comparaison ensembliste** sur `pair_key` normalisés :
   - TP = accord regex ∩ LLM  
   - FP_regex = regex seul (possibles faux positifs / normalisations divergentes)  
   - FN_regex = LLM seul (formulations que le regex rate)
5. **Fiche de calibration** : precision / recall / F1 global + par juridiction + catégorisation qualitative des écarts.

---
**Prérequis cluster.**  
- Setup cluster identique à `benchmark_m1_m6_sample.ipynb`.  
- Les données `database-judilibre-enrichie/{Cour de cassation, Cours d'appel, Tribunal judiciaire}` doivent être accessibles depuis le node.
- Ajuster `DATA_ENRICHED_DIR` ci-dessous au chemin réel.

---
## 0. Setup cluster — install, download, démarrage vLLM

**Cellules 0.2 → 0.6 copiées de `benchmark_m1_m6_sample.ipynb`.** À exécuter une fois par session.

In [1]:
# ═══════════════════════════════════════════════════════════════════════
# 0.2 — CONFIG CLUSTER
# ═══════════════════════════════════════════════════════════════════════
import os
import subprocess
from pathlib import Path

# ── Modèle ────────────────────────────────────────────────────────────
# Qwen2.5-14B-Instruct-AWQ : quantifié 4-bit par l'équipe Qwen
#   - Poids ~10 Go (tient très à l'aise sur 1 × L40S 48 Go)
#   - Qualité quasi-identique au bf16 sur extraction structurée
#   - Permet de garder MAX_LEN=32768 avec GPU_UTIL=0.90 sans stress
#
# Alternatives (si besoin) :
#   - "Qwen/Qwen2.5-14B-Instruct"          → bf16, ~30 Go (serré sur L40S, viser MAX_LEN 16k-24k)
#   - "Qwen/Qwen2.5-7B-Instruct"           → bf16 plus léger (~15 Go), qualité moindre
#   - "Qwen/Qwen2.5-7B-Instruct-1M"        → contexte 1M, VRAM + importante
MODEL_ID  = "Qwen/Qwen2.5-14B-Instruct-AWQ"

VLLM_PORT = 8000

# Contexte max natif Qwen2.5 = 32768 tokens.
# Avec l'AWQ on a largement assez de VRAM pour 32k sur L40S.
MAX_LEN   = 32768
GPU_UTIL  = 0.90

HF_TOKEN = os.environ.get("HF_TOKEN")

try:
    out = subprocess.check_output(
        ["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv,noheader"],
        text=True,
    )
    gpus = [l for l in out.strip().split("\n") if l]
    NUM_GPUS = len(gpus)
    print(f"GPUs détectés : {NUM_GPUS}")
    for i, g in enumerate(gpus):
        print(f"  [{i}] {g}")
except Exception as e:
    print(f"[WARN] nvidia-smi indisponible ({e}) → NUM_GPUS=1")
    NUM_GPUS = 1

LOG_DIR = Path("./logs"); LOG_DIR.mkdir(exist_ok=True)
VLLM_LOG = LOG_DIR / "vllm_server.log"
VLLM_PID = LOG_DIR / "vllm.pid"

VLLM_BASE_URL = f"http://localhost:{VLLM_PORT}/v1"
print(f"\nModèle        : {MODEL_ID}")
print(f"Contexte max  : {MAX_LEN} tokens  (~{MAX_LEN*3//1000}k caractères)")
print(f"vLLM serveur  : {VLLM_BASE_URL}")

GPUs détectés : 1
  [0] NVIDIA L40S, 46068 MiB

Modèle        : Qwen/Qwen2.5-14B-Instruct-AWQ
Contexte max  : 32768 tokens  (~98k caractères)
vLLM serveur  : http://localhost:8000/v1


In [2]:
# ═══════════════════════════════════════════════════════════════════════
# 0.3 — INSTALLATION DES DÉPENDANCES
# Bootstrap pip, écrase le scipy/cryptography système cassé, puis installe vllm.
# ═══════════════════════════════════════════════════════════════════════
import subprocess, sys, importlib

def _ensure_pip():
    try:
        subprocess.check_call(
            [sys.executable, "-m", "pip", "--version"],
            stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL,
        )
        return
    except (subprocess.CalledProcessError, FileNotFoundError):
        pass
    print("[bootstrap] pip absent — installation via ensurepip …")
    subprocess.check_call([sys.executable, "-m", "ensurepip", "--upgrade"])
    subprocess.check_call(
        [sys.executable, "-m", "pip", "install", "--quiet", "--upgrade", "pip"]
    )
    print("[bootstrap] pip installé.")

_ensure_pip()

# ── 1) Stack numérique compatible (évite scipy système cassé) ─────────
CORE_NUMERIC = [
    "numpy>=1.26,<2",
    "scipy>=1.11,<1.14",
]
print("1/3 — numpy/scipy compatibles (écrase le système cassé) …")
subprocess.check_call(
    [sys.executable, "-m", "pip", "install", "--quiet", "--upgrade",
     "--force-reinstall", "--no-deps", *CORE_NUMERIC]
)

# ── 2) Stack crypto moderne (évite cryptography/pyOpenSSL système cassés) ──
# Le système a un cryptography._rust qui cherche un _cffi_backend absent.
# On force une version fraîche dans le venv : cffi + cryptography + pyOpenSSL
# ainsi que boto3/urllib3 qui sont dans la chaîne d'import de vLLM.
CRYPTO_STACK = [
    "cffi>=1.17",
    "cryptography>=42",
    "pyOpenSSL>=24",
    "urllib3>=1.26,<3",
    "boto3>=1.34",      # vllm.transformers_utils.s3_utils l'importe
    "botocore>=1.34",
]
print("2/3 — stack crypto moderne (écrase cryptography/pyOpenSSL système) …")
subprocess.check_call(
    [sys.executable, "-m", "pip", "install", "--quiet", "--upgrade",
     "--force-reinstall", *CRYPTO_STACK]
)

# ── 3) Dépendances applicatives ───────────────────────────────────────
DEPS = [
    "vllm>=0.8.5",
    "openai>=1.50",
    "pydantic>=2",
    "pandas>=2",
    "pyarrow>=14",
    "huggingface_hub>=0.25",
    "tqdm>=4.65",
    "rich>=13",
    "tabulate>=0.9",
]
print("3/3 — vllm et autres dépendances …")
subprocess.check_call([sys.executable, "-m", "pip", "install", "--quiet", *DEPS])
importlib.invalidate_caches()

# ── Sanity checks ─────────────────────────────────────────────────────
print("\n── Sanity checks ──")
import numpy, scipy
print(f"✓ numpy {numpy.__version__}  ·  scipy {scipy.__version__}")
try:
    from scipy.optimize import linear_sum_assignment
    print("✓ scipy.optimize OK")
except Exception as e:
    print(f"[WARN] scipy : {e}")
try:
    import cffi, cryptography, OpenSSL
    print(f"✓ cffi {cffi.__version__}  ·  cryptography {cryptography.__version__}  "
          f"·  pyOpenSSL {OpenSSL.__version__}")
    # Le chemin doit pointer vers .local ou venv, PAS vers /usr/lib
    print(f"  cryptography from : {cryptography.__file__}")
except Exception as e:
    print(f"[WARN] crypto : {e}")
try:
    import boto3
    print(f"✓ boto3 {boto3.__version__}  ·  from : {boto3.__file__}")
except Exception as e:
    print(f"[WARN] boto3 : {e}")

print("\n✓ Dépendances installées. Continuer avec la cellule suivante.")

1/3 — numpy/scipy compatibles (écrase le système cassé) …
2/3 — stack crypto moderne (écrase cryptography/pyOpenSSL système) …


ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
jupyter 1.1.1 requires jupyterlab, which is not installed.
jupyter 1.1.1 requires nbconvert, which is not installed.
jupyter 1.1.1 requires notebook, which is not installed.
matplotlib 3.10.8 requires kiwisolver>=1.3.1, but you have kiwisolver 0.0.0 which is incompatible.


3/3 — vllm et autres dépendances …


ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
datasets 4.8.4 requires requests>=2.32.2, but you have requests 2.31.0 which is incompatible.
matplotlib 3.10.8 requires kiwisolver>=1.3.1, but you have kiwisolver 0.0.0 which is incompatible.



── Sanity checks ──
✓ numpy 2.2.6  ·  scipy 1.13.1
✓ scipy.optimize OK
✓ cffi 2.0.0  ·  cryptography 46.0.7  ·  pyOpenSSL 26.0.0
  cryptography from : /home/ids/kaeppelin-22/.local/lib/python3.10/site-packages/cryptography/__init__.py
✓ boto3 1.42.94  ·  from : /home/ids/kaeppelin-22/.local/lib/python3.10/site-packages/boto3/__init__.py

✓ Dépendances installées. Continuer avec la cellule suivante.


In [3]:
# ═══════════════════════════════════════════════════════════════════════
# 0.4 — AUTH HUGGINGFACE + DOWNLOAD DU MODÈLE
# Self-healing : bootstrap pip puis réinstalle le package manquant si besoin.
# ═══════════════════════════════════════════════════════════════════════
import importlib, subprocess, sys

def _ensure_pip():
    try:
        subprocess.check_call(
            [sys.executable, "-m", "pip", "--version"],
            stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL,
        )
    except (subprocess.CalledProcessError, FileNotFoundError):
        print("[bootstrap] pip absent — installation via ensurepip …")
        subprocess.check_call([sys.executable, "-m", "ensurepip", "--upgrade"])

def _ensure(pkg: str, import_name: str | None = None):
    name = import_name or pkg.split(">")[0].split("=")[0].strip()
    try:
        importlib.import_module(name)
        return
    except ModuleNotFoundError:
        pass
    _ensure_pip()
    print(f"[fix] Installation de {pkg} …")
    subprocess.check_call([sys.executable, "-m", "pip", "install", "--quiet", pkg])
    importlib.invalidate_caches()

_ensure("huggingface_hub>=0.25", "huggingface_hub")

from huggingface_hub import login, snapshot_download

if HF_TOKEN:
    login(token=HF_TOKEN, add_to_git_credential=False)
    print("✓ Authentifié sur HuggingFace")
else:
    print("[WARN] HF_TOKEN non défini — exporter HF_TOKEN=hf_xxx avant de lancer Jupyter.")

print(f"\nTéléchargement de {MODEL_ID}…")
model_path = snapshot_download(repo_id=MODEL_ID, ignore_patterns=["*.md", "*.txt", "original/*"])
print(f"\n✓ Modèle en cache : {model_path}")

[WARN] HF_TOKEN non défini — exporter HF_TOKEN=hf_xxx avant de lancer Jupyter.

Téléchargement de Qwen/Qwen2.5-14B-Instruct-AWQ…


Fetching 11 files:   0%|          | 0/11 [00:00<?, ?it/s]


✓ Modèle en cache : /home/ids/kaeppelin-22/.cache/huggingface/hub/models--Qwen--Qwen2.5-14B-Instruct-AWQ/snapshots/539535859b135b0244c91f3e59816150c8056698


In [4]:
# ═══════════════════════════════════════════════════════════════════════
# 0.5 — DÉMARRAGE SERVEUR vLLM
# On invoque `python -m vllm.entrypoints.openai.api_server` plutôt que
# la CLI globale `vllm`, pour garantir qu'on utilise le vLLM installé
# dans le venv du kernel courant (évite le ModuleNotFoundError si la CLI
# /home/.../.local/bin/vllm pointe vers un autre Python).
# ═══════════════════════════════════════════════════════════════════════
import subprocess, os, signal, sys, time

if VLLM_PID.exists():
    try:
        old = int(VLLM_PID.read_text())
        os.killpg(os.getpgid(old), signal.SIGTERM)
        print(f"Serveur précédent (PID={old}) arrêté")
        time.sleep(3)
    except (ProcessLookupError, ValueError, PermissionError):
        pass

cmd = [
    sys.executable, "-m", "vllm.entrypoints.openai.api_server",
    "--model", MODEL_ID,
    "--tensor-parallel-size", str(NUM_GPUS),
    "--max-model-len", str(MAX_LEN),
    "--gpu-memory-utilization", str(GPU_UTIL),
    "--port", str(VLLM_PORT),
]
print("Commande :", " ".join(cmd))

log_f = open(VLLM_LOG, "w")
vllm_proc = subprocess.Popen(cmd, stdout=log_f, stderr=subprocess.STDOUT, preexec_fn=os.setsid)
VLLM_PID.write_text(str(vllm_proc.pid))
print(f"✓ vLLM démarré (PID={vllm_proc.pid})  ·  logs → {VLLM_LOG}")

Commande : /usr/bin/python3.10 -m vllm.entrypoints.openai.api_server --model Qwen/Qwen2.5-14B-Instruct-AWQ --tensor-parallel-size 1 --max-model-len 32768 --gpu-memory-utilization 0.9 --port 8000
✓ vLLM démarré (PID=2430692)  ·  logs → logs/vllm_server.log


In [5]:
# ═══════════════════════════════════════════════════════════════════════
# 0.6 — ATTENTE DU DÉMARRAGE
# ═══════════════════════════════════════════════════════════════════════
import time, urllib.request, json

HEALTH_URL = f"http://localhost:{VLLM_PORT}/health"
MAX_WAIT_S = 900

t0 = time.time()
ready = False
while time.time() - t0 < MAX_WAIT_S:
    if vllm_proc.poll() is not None:
        raise RuntimeError(f"vLLM s'est arrêté (code={vllm_proc.returncode}) — inspecter {VLLM_LOG}")
    try:
        with urllib.request.urlopen(HEALTH_URL, timeout=2) as r:
            if r.status == 200:
                ready = True; break
    except Exception:
        pass
    print(f"  chargement… {int(time.time()-t0)}s (max {MAX_WAIT_S}s)", end="\r")
    time.sleep(5)

if not ready:
    raise RuntimeError(f"vLLM non prêt après {MAX_WAIT_S}s — voir {VLLM_LOG}")

print(f"\n✓ vLLM prêt  (démarrage : {int(time.time()-t0)}s)")
with urllib.request.urlopen(f"http://localhost:{VLLM_PORT}/v1/models") as r:
    data = json.load(r)
print("Modèles :", [m['id'] for m in data.get('data', [])])

  chargement… 55s (max 900s)
✓ vLLM prêt  (démarrage : 60s)
Modèles : ['Qwen/Qwen2.5-14B-Instruct-AWQ']


---
## 1. Configuration du benchmark de validation

In [6]:
# ═══════════════════════════════════════════════════════════════════════
# CONFIG VALIDATION REGEX
# ═══════════════════════════════════════════════════════════════════════
API_KEY       = "EMPTY"                       # vLLM n'impose pas de clé

# ─────────────────────────────────────────────────────────────────────
# SOURCE DES ARRÊTS
# Option A (RECOMMANDÉE sur cluster) — JSONL pré-échantillonné uploadé
#   Fichier produit localement par `prepare_regex_validation_sample.py`
# Option B (développement local) — scan complet des 3 datasets JSONL
# ─────────────────────────────────────────────────────────────────────
PRESAMPLED_PATH   = Path("./cluster_data/regex_validation/sample_100.jsonl")
DATA_ENRICHED_DIR = Path("./database-judilibre-enrichie")   # fallback Option B

RESULTS_DIR   = Path("./results"); RESULTS_DIR.mkdir(exist_ok=True)

# Sampling (utilisé uniquement en Option B)
N_TOTAL       = 100
N_CC          = 50
N_CA          = 25
N_TJ          = 25
SEED          = 42
MIN_TEXT_LEN  = 500

# LLM
TEMPERATURE      = 0.0
MAX_TOKENS_OUT   = 4096

# Garde-fou longueur du prompt utilisateur (texte de l'arrêt).
# MAX_LEN côté vLLM = 32 768 tokens. On prévoit :
#   - MAX_TOKENS_OUT (4 096) pour la sortie
#   - ~500 tokens pour le system prompt + framing
#   - le reste pour l'arrêt : ~28 000 tokens
# À 3 car/token pour le français : ~84 000 caractères safe.
MAX_PROMPT_CHARS = 80_000

DATASET_PATHS = {
    "CC": DATA_ENRICHED_DIR / "Cour de cassation",
    "CA": DATA_ENRICHED_DIR / "Cours d'appel",
    "TJ": DATA_ENRICHED_DIR / "Tribunal judiciaire",
}

# Auto-détection du mode
USE_PRESAMPLED = PRESAMPLED_PATH.exists()
if USE_PRESAMPLED:
    print(f"✓ Mode A — JSONL pré-échantillonné : {PRESAMPLED_PATH}  "
          f"({PRESAMPLED_PATH.stat().st_size/1e6:.2f} Mo)")
else:
    print(f"✗ Pas de fichier pré-échantillonné ({PRESAMPLED_PATH})")
    print(f"  → Mode B : sampling depuis les datasets complets")
    for k, p in DATASET_PATHS.items():
        status = "✓" if p.exists() else "✗ INTROUVABLE"
        print(f"  {k}: {p}  [{status}]")

✓ Mode A — JSONL pré-échantillonné : cluster_data/regex_validation/sample_100.jsonl  (1.25 Mo)


---
## 2. Imports + schémas + fonctions de normalisation

Les fonctions `normalize_code` et `normalize_article` reproduisent à l'identique celles du notebook `enrichissement_base_complete.ipynb` pour que les `pair_keys` du LLM et du regex soient comparables sur la même convention canonique.

In [7]:
import json, re, time, random, unicodedata
from collections import Counter, defaultdict
from typing import Optional

import pandas as pd
from pydantic import BaseModel, Field
from openai import OpenAI

try:
    from tqdm.notebook import tqdm
except ImportError:
    from tqdm import tqdm

pd.set_option("display.max_colwidth", None)
pd.set_option("display.max_rows", 200)
pd.set_option("display.width", None)

In [8]:
# ═══════════════════════════════════════════════════════════════════════
# NORMALISATION canonique (identique à enrichissement_base_complete.ipynb)
# ═══════════════════════════════════════════════════════════════════════

def _strip_accents(s: str) -> str:
    return "".join(
        c for c in unicodedata.normalize("NFD", s)
        if unicodedata.category(c) != "Mn"
    )

def normalize_code(name: str) -> str:
    """'Code civil' -> 'code_civil', 'Code du travail' -> 'code_du_travail'."""
    s = name.lower().strip()
    s = _strip_accents(s)
    s = re.sub(r"[\u2018\u2019']", "_", s)
    s = re.sub(r"[^a-z0-9]+", "_", s)
    s = re.sub(r"_+", "_", s).strip("_")
    return s

def normalize_article(prefix: str, number: str) -> str:
    prefix = prefix.upper().strip() if prefix else ""
    num = re.sub(r"[\s.]+", "", number) if number else ""
    if prefix:
        return f"{prefix}{num}"
    return num

def make_pair_key(code_slug: str, article_num: str) -> str:
    return f"{code_slug}:{article_num}"

# Sanity checks
assert normalize_code("Code civil") == "code_civil"
assert normalize_code("Code du travail") == "code_du_travail"
assert normalize_article("L", "122-14-3") == "L122-14-3"
assert normalize_article("", "1240") == "1240"
assert make_pair_key("code_civil", "1240") == "code_civil:1240"
print("✓ Fonctions de normalisation validées")

✓ Fonctions de normalisation validées


In [9]:
# ═══════════════════════════════════════════════════════════════════════
# SCHÉMA PYDANTIC pour la sortie LLM
# ═══════════════════════════════════════════════════════════════════════

class ArticleCitation(BaseModel):
    """Un article cité dans l'arrêt."""
    code_name: str = Field(
        description="Nom du code exactement comme écrit dans le texte (ex: 'Code civil', 'Code du travail', 'Code général des impôts')."
    )
    article_number: str = Field(
        description="Numéro de l'article tel qu'il apparaît (ex: '1240', 'L. 122-14-3', 'R. 431-5', 'A. 302 septies A')."
    )
    verbatim: str = Field(
        description="Extrait du texte où l'article est cité (max 200 caractères)."
    )

class ArticleExtractionOutput(BaseModel):
    """Liste des articles extraits d'un arrêt."""
    articles: list[ArticleCitation] = Field(
        description="Toutes les occurrences d'articles de loi cités dans l'arrêt (avec code ET numéro). Chaque occurrence distincte est listée une fois."
    )

print("✓ Schéma Pydantic défini")

✓ Schéma Pydantic défini


---
## 3. Sampling stratifié 50 CC + 25 CA + 25 TJ

In [10]:
# ═══════════════════════════════════════════════════════════════════════
# CHARGEMENT / SAMPLING DES ARRÊTS
# Mode A : on lit directement le JSONL pré-échantillonné uploadé
# Mode B : on fait un reservoir sampling sur les 3 datasets complets
# ═══════════════════════════════════════════════════════════════════════

def load_presampled(path: Path) -> list[tuple[str, dict]]:
    """Charge le JSONL produit par `prepare_regex_validation_sample.py`.
    Chaque ligne est un record Judilibre avec un champ `_jurisdiction` ajouté.
    """
    out: list[tuple[str, dict]] = []
    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            rec = json.loads(line)
            jur = rec.pop("_jurisdiction", "??")
            out.append((jur, rec))
    return out


def reservoir_sample(path: Path, k: int, min_text_len: int, seed: int) -> list[dict]:
    """Reservoir sampling sur un fichier JSONL (Option B)."""
    rng = random.Random(seed)
    pool: list[dict] = []
    seen_valid = 0
    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            try:
                rec = json.loads(line)
            except json.JSONDecodeError:
                continue
            if not rec.get("text") or len(rec["text"]) < min_text_len:
                continue
            seen_valid += 1
            if len(pool) < k:
                pool.append(rec)
            else:
                j = rng.randint(0, seen_valid - 1)
                if j < k:
                    pool[j] = rec
    return pool


if USE_PRESAMPLED:
    print(f"Chargement : {PRESAMPLED_PATH}")
    samples = load_presampled(PRESAMPLED_PATH)
    # Compteurs par juridiction
    from collections import Counter
    jur_counts = Counter(j for j, _ in samples)
    for jur in ["CC", "CA", "TJ"]:
        print(f"  {jur} : {jur_counts.get(jur, 0):>3} arrêts")
    print(f"Total : {len(samples)} arrêts chargés depuis le JSONL pré-échantillonné.")
else:
    print("Sampling en cours (Mode B, reservoir sampling sur les 3 datasets)…")
    t0 = time.time()
    sample_cc = reservoir_sample(DATASET_PATHS["CC"], N_CC, MIN_TEXT_LEN, SEED + 1)
    print(f"  CC : {len(sample_cc):>3} arrêts  ({time.time()-t0:.1f}s)")
    sample_ca = reservoir_sample(DATASET_PATHS["CA"], N_CA, MIN_TEXT_LEN, SEED + 2)
    print(f"  CA : {len(sample_ca):>3} arrêts  ({time.time()-t0:.1f}s cumul.)")
    sample_tj = reservoir_sample(DATASET_PATHS["TJ"], N_TJ, MIN_TEXT_LEN, SEED + 3)
    print(f"  TJ : {len(sample_tj):>3} arrêts  ({time.time()-t0:.1f}s cumul.)")

    samples = (
        [("CC", r) for r in sample_cc] +
        [("CA", r) for r in sample_ca] +
        [("TJ", r) for r in sample_tj]
    )
    print(f"\nTotal : {len(samples)} arrêts échantillonnés.")

Chargement : cluster_data/regex_validation/sample_100.jsonl
  CC :  50 arrêts
  CA :  25 arrêts
  TJ :  25 arrêts
Total : 100 arrêts chargés depuis le JSONL pré-échantillonné.


In [11]:
# Récap du sample : distribution par juridiction, longueur, code_article_pairs déjà extraits
import statistics

for jur in ["CC", "CA", "TJ"]:
    sub = [r for j, r in samples if j == jur]
    text_lens = [len(r.get("text") or "") for r in sub]
    n_with_pairs = sum(1 for r in sub if r.get("code_article_pairs"))
    n_pairs_total = sum(len(r.get("code_article_pairs") or []) for r in sub)
    print(f"─ {jur} ─ n={len(sub)}")
    print(f"    text len : min={min(text_lens)}  med={int(statistics.median(text_lens))}  max={max(text_lens)}")
    print(f"    code_article_pairs : {n_with_pairs}/{len(sub)} arrêts avec ≥1 paire ({n_pairs_total} paires totales)")

# ═══════════════════════════════════════════════════════════════════════
# SANITY CHECK LONGUEUR — s'assurer que le LLM pourra voir le texte intégral
# ═══════════════════════════════════════════════════════════════════════
print("\n── Contrôle longueur vs MAX_PROMPT_CHARS ──")
all_text_lens = [(jur, len(r.get("text") or ""), r.get("id")) for jur, r in samples]
all_text_lens.sort(key=lambda x: -x[1])

n_oversized = sum(1 for _, l, _ in all_text_lens if l > MAX_PROMPT_CHARS)
print(f"MAX_PROMPT_CHARS = {MAX_PROMPT_CHARS:,}  |  Arrêts qui dépassent : {n_oversized}/{len(all_text_lens)}")

print("\nTop 10 des arrêts les plus longs :")
for jur, l, id_ in all_text_lens[:10]:
    flag = "⚠️ TRONCATURE" if l > MAX_PROMPT_CHARS else "✓"
    print(f"  {jur}  len={l:>8,}  {flag}  id={id_}")

if n_oversized > 0:
    print(f"\n[WARN] {n_oversized} arrêts seront tronqués à {MAX_PROMPT_CHARS} car "
          f"→ flag `text_truncated=True` dans les résultats. Si gênant, augmenter "
          f"MAX_LEN + MAX_PROMPT_CHARS et redémarrer le serveur vLLM.")

─ CC ─ n=50
    text len : min=870  med=3381  max=32725
    code_article_pairs : 45/50 arrêts avec ≥1 paire (99 paires totales)
─ CA ─ n=25
    text len : min=1636  med=9974  max=112579
    code_article_pairs : 24/25 arrêts avec ≥1 paire (82 paires totales)
─ TJ ─ n=25
    text len : min=3249  med=10782  max=44876
    code_article_pairs : 23/25 arrêts avec ≥1 paire (135 paires totales)

── Contrôle longueur vs MAX_PROMPT_CHARS ──
MAX_PROMPT_CHARS = 80,000  |  Arrêts qui dépassent : 1/100

Top 10 des arrêts les plus longs :
  CA  len= 112,579  ⚠️ TRONCATURE  id=6033618699c14d1285657add
  TJ  len=  44,876  ✓  id=673cf442956b912a0059c641
  CA  len=  40,155  ✓  id=633d1fbd62f5393e2eb448e9
  TJ  len=  33,315  ✓  id=66562596f76bcc1332d0eb50
  TJ  len=  33,028  ✓  id=66561fc3f76bcc1332cfbdf6
  CC  len=  32,725  ✓  id=5fd946f217fac52e4f8a1ccb
  CA  len=  25,766  ✓  id=63806c3859a9bf05d40aca82
  CC  len=  25,233  ✓  id=5fca4af1efd5034a37e08617
  CA  len=  22,514  ✓  id=665968aa378099000886537b


---
## 4. Baseline regex — lue depuis `code_article_pairs` des records enrichis

Le champ `code_article_pairs` contient **exactement la sortie du regex** tel qu'il tourne en production. On se contente de le lire et de le normaliser (il l'est déjà, mais on sécurise).

In [12]:
def regex_pair_keys(rec: dict) -> set[str]:
    """Retourne l'ensemble des pair_keys extraits par le regex (du corpus enrichi)."""
    pairs = rec.get("code_article_pairs") or []
    out = set()
    for p in pairs:
        if not p or ":" not in p:
            continue
        slug, num = p.split(":", 1)
        out.add(make_pair_key(slug.strip(), num.strip()))
    return out

# Calcul sur l'échantillon
regex_pairs_per_sample = [(jur, rec, regex_pair_keys(rec)) for jur, rec in samples]
sizes = [len(pk) for _, _, pk in regex_pairs_per_sample]
print(f"pair_keys regex par arrêt : min={min(sizes)}  med={int(statistics.median(sizes))}  max={max(sizes)}  moy={sum(sizes)/len(sizes):.1f}")
print(f"Total unique regex pair_keys sur l'échantillon : {len({pk for _,_,pks in regex_pairs_per_sample for pk in pks})}")

pair_keys regex par arrêt : min=0  med=2  max=12  moy=3.2
Total unique regex pair_keys sur l'échantillon : 184


---
## 5. Vérification serveur + fonction d'appel LLM structuré

Reprise à l'identique de `benchmark_m1_m6_sample.ipynb` (appel vLLM avec `response_format` JSON strict).

In [13]:
client = OpenAI(base_url=VLLM_BASE_URL, api_key=API_KEY)

try:
    models = client.models.list()
    print("Serveur OK — modèles :", [m.id for m in models.data])
    assert MODEL_ID in [m.id for m in models.data]
except Exception as e:
    raise RuntimeError(f"Serveur vLLM indisponible : {e}")

Serveur OK — modèles : ['Qwen/Qwen2.5-14B-Instruct-AWQ']


In [14]:
def call_llm_structured(
    system_prompt: str,
    user_prompt: str,
    output_model: type[BaseModel],
    max_tokens: int = MAX_TOKENS_OUT,
) -> tuple[Optional[BaseModel], dict]:
    """Appel vLLM avec sortie JSON structurée."""
    t0 = time.time()
    meta = {
        "latency_s": None, "tokens_used": None, "finish_reason": None,
        "error": None, "raw_response": None,
    }
    try:
        resp = client.chat.completions.create(
            model=MODEL_ID,
            messages=[
                {"role": "system", "content": system_prompt},
                {"role": "user",   "content": user_prompt},
            ],
            temperature=TEMPERATURE,
            max_tokens=max_tokens,
            response_format={
                "type": "json_schema",
                "json_schema": {
                    "name": output_model.__name__,
                    "schema": output_model.model_json_schema(),
                    "strict": True,
                },
            },
        )
        raw = resp.choices[0].message.content
        meta["latency_s"]     = round(time.time() - t0, 2)
        meta["tokens_used"]   = resp.usage.completion_tokens if resp.usage else None
        meta["finish_reason"] = resp.choices[0].finish_reason
        meta["raw_response"]  = raw

        if meta["finish_reason"] == "length":
            meta["error"] = f"Coupé par max_tokens={max_tokens}"
            return None, meta

        json_match = re.search(r'\{.*\}', raw, re.DOTALL)
        json_str = json_match.group(0) if json_match else raw
        result = output_model.model_validate_json(json_str)
        return result, meta
    except Exception as e:
        meta["latency_s"] = round(time.time() - t0, 2)
        meta["error"] = f"{type(e).__name__}: {e}"
        return None, meta

---
## 6. Prompt d'extraction + boucle sur les 100 arrêts

In [15]:
SYSTEM_PROMPT = """Tu es un juriste français expert. Ta tâche est d'extraire de façon EXHAUSTIVE toutes les citations d'articles de loi présentes dans un arrêt de justice.

## RÈGLES

1. Un "article" doit avoir un NUMÉRO explicite (ex: 1240, L. 122-14-3, R. 431-5, 302 septies A).
2. Tu dois identifier le CODE auquel il appartient (Code civil, Code du travail, Code de procédure civile, Code pénal, Code général des impôts, etc.). Si le code n'est pas explicitement mentionné mais qu'il est clairement inférable du contexte, tu peux l'indiquer.
3. Si un article est cité SANS code (ex: simplement "article 700") et que le code n'est pas évident depuis le contexte, NE PAS l'inclure.
4. Chaque occurrence DISTINCTE d'un couple (code, article) est listée UNE FOIS (pas de doublon même si cité plusieurs fois).
5. Pour les articles en forme compacte (L1234-5), reprends la forme originale du texte.
6. `verbatim` : extrait texte exact (max 200 car) qui contient la citation.
7. L'arrêt peut être LONG. Parcours-le intégralement — notamment la zone "visa", les moyens, les motivations, le dispositif et les annexes.

## EXEMPLES

- "Vu l'article 1240 du Code civil" → code_name="Code civil", article_number="1240"
- "l'article L. 1121-1 du code du travail" → code_name="Code du travail", article_number="L. 1121-1"
- "l'article 700" (sans code) → NE PAS inclure
- "en vertu du Code civil, article 1382 ancien" → code_name="Code civil", article_number="1382"

Réponds UNIQUEMENT avec le JSON demandé, sans préambule."""


def build_user_prompt(text: str) -> tuple[str, bool]:
    """Construit le prompt utilisateur. Passe le texte INTÉGRAL à moins qu'il ne
    dépasse MAX_PROMPT_CHARS (garde-fou contextuel).

    Returns
    -------
    (prompt, truncated) : str, bool
        prompt    — le prompt utilisateur complet
        truncated — True si le texte a dû être tronqué (à logger)
    """
    truncated = len(text) > MAX_PROMPT_CHARS
    t = text[:MAX_PROMPT_CHARS] if truncated else text
    suffix = "\n\n[... texte tronqué — reste de l'arrêt non transmis]" if truncated else ""
    return (
        f"Extrais tous les couples (code, article) cités dans cet arrêt :\n\n---\n{t}{suffix}\n---",
        truncated,
    )


print("✓ Prompt défini (pas de troncature sauf si > MAX_PROMPT_CHARS).")

✓ Prompt défini (pas de troncature sauf si > MAX_PROMPT_CHARS).


In [ ]:
# ═══════════════════════════════════════════════════════════════════════
# EXTRACTION LLM sur les 100 arrêts
# ═══════════════════════════════════════════════════════════════════════

llm_results = []  # un dict par arrêt

for jur, rec, regex_pks in tqdm(regex_pairs_per_sample, desc="LLM extraction"):
    text = rec.get("text") or ""
    user_prompt, truncated = build_user_prompt(text)
    output, meta = call_llm_structured(
        system_prompt=SYSTEM_PROMPT,
        user_prompt=user_prompt,
        output_model=ArticleExtractionOutput,
        max_tokens=MAX_TOKENS_OUT,
    )

    llm_pairs_raw: list[dict] = []
    llm_pks: set[str] = set()

    if output is not None:
        for art in output.articles:
            slug = normalize_code(art.code_name)
            num_match = re.match(r"^\s*([LRDAE])?\.?\s*(.+?)\s*$", art.article_number, re.IGNORECASE)
            if num_match:
                prefix = num_match.group(1) or ""
                number = num_match.group(2)
            else:
                prefix = ""; number = art.article_number
            art_norm = normalize_article(prefix, number)
            pk = make_pair_key(slug, art_norm)
            if slug and art_norm:
                llm_pks.add(pk)
            llm_pairs_raw.append({
                "code_name": art.code_name,
                "article_number": art.article_number,
                "verbatim": art.verbatim,
                "pair_key_norm": pk,
            })

    llm_results.append({
        "jurisdiction":   jur,
        "id":             rec.get("id"),
        "ecli":           rec.get("ecli"),
        "date":           rec.get("decision_date"),
        "chamber":        rec.get("chamber"),
        "text_len":       len(text),
        "text_truncated": truncated,   # True si on a dû tronquer pour le LLM
        "regex_pks":      sorted(regex_pks),
        "llm_pks":        sorted(llm_pks),
        "llm_pairs_raw":  llm_pairs_raw,
        "latency_s":      meta["latency_s"],
        "tokens_used":    meta["tokens_used"],
        "finish_reason":  meta["finish_reason"],
        "error":          meta["error"],
    })

n_trunc = sum(1 for r in llm_results if r.get("text_truncated"))
n_err   = sum(1 for r in llm_results if r["error"])
print(f"\n✓ {len(llm_results)} arrêts traités  ·  erreurs : {n_err}  ·  tronqués : {n_trunc}")

LLM extraction:   0%|          | 0/100 [00:00<?, ?it/s]

---
## 7. Comparaison regex ↔ LLM — métriques globales

In [ ]:
# ═══════════════════════════════════════════════════════════════════════
# MÉTRIQUES par arrêt
# ═══════════════════════════════════════════════════════════════════════

rows = []
for r in llm_results:
    R = set(r["regex_pks"])
    L = set(r["llm_pks"])
    tp = R & L
    fp_regex = R - L     # regex seul (peut être un faux positif regex ou un oubli LLM)
    fn_regex = L - R     # LLM seul (regex a raté)

    precision = len(tp) / max(1, len(R))   # précision du regex vs LLM-référence
    recall    = len(tp) / max(1, len(L))   # recall du regex vs LLM-référence
    f1 = 2*precision*recall / (precision+recall) if (precision+recall) > 0 else 0.0

    rows.append({
        "jurisdiction":   r["jurisdiction"],
        "id":             r["id"],
        "date":           r["date"],
        "chamber":        r["chamber"],
        "text_len":       r["text_len"],
        "text_truncated": r.get("text_truncated", False),
        "n_regex":        len(R),
        "n_llm":          len(L),
        "n_tp":           len(tp),
        "n_fp_regex":     len(fp_regex),
        "n_fn_regex":     len(fn_regex),
        "precision":      round(precision, 3),
        "recall":         round(recall, 3),
        "f1":             round(f1, 3),
        "latency_s":      r["latency_s"],
        "error":          r["error"],
    })

df = pd.DataFrame(rows)
print(f"DataFrame : {len(df)} lignes  ·  tronqués : {df['text_truncated'].sum()}")
df.head(10)

In [ ]:
# ═══════════════════════════════════════════════════════════════════════
# MÉTRIQUES AGRÉGÉES
# ═══════════════════════════════════════════════════════════════════════

print("═" * 70)
print(f"RÉSULTATS VALIDATION REGEX  ({len(df)} arrêts)")
print("═" * 70)

df_ok = df[df["error"].isna()]
print(f"\nArrêts LLM valides : {len(df_ok)}/{len(df)}")

# Pondération ensemble : vrai pool de pair_keys (pas moyenne par arrêt, pour éviter le biais des petits arrêts)
TP = df_ok["n_tp"].sum()
FP = df_ok["n_fp_regex"].sum()
FN = df_ok["n_fn_regex"].sum()

if TP + FP > 0 and TP + FN > 0:
    P = TP / (TP + FP)
    R = TP / (TP + FN)
    F1 = 2*P*R / (P + R) if (P+R) > 0 else 0.0
else:
    P = R = F1 = 0.0

print(f"\n▸ Métriques globales (pooled sur tous les pair_keys)")
print(f"    Total pair_keys regex : {df_ok['n_regex'].sum()}")
print(f"    Total pair_keys LLM   : {df_ok['n_llm'].sum()}")
print(f"    TP                    : {TP}")
print(f"    FP (regex seul)       : {FP}")
print(f"    FN (LLM seul, regex a raté) : {FN}")
print(f"    Précision  regex vs LLM    : {P:.3f}")
print(f"    Recall     regex vs LLM    : {R:.3f}")
print(f"    F1                         : {F1:.3f}")

print(f"\n▸ Par juridiction")
by_jur = df_ok.groupby("jurisdiction").agg(
    n=("id", "count"),
    n_regex=("n_regex", "sum"),
    n_llm=("n_llm", "sum"),
    tp=("n_tp", "sum"),
    fp=("n_fp_regex", "sum"),
    fn=("n_fn_regex", "sum"),
)
by_jur["precision"] = (by_jur["tp"] / (by_jur["tp"] + by_jur["fp"])).round(3)
by_jur["recall"]    = (by_jur["tp"] / (by_jur["tp"] + by_jur["fn"])).round(3)
by_jur["f1"]        = (2*by_jur["precision"]*by_jur["recall"] / (by_jur["precision"]+by_jur["recall"])).round(3)
print(by_jur.to_string())

---
## 8. Catégorisation qualitative des écarts (FN du regex)

Pour chaque FN du regex (pair_key trouvé par le LLM mais pas par le regex), on inspecte le `verbatim` pour comprendre **pourquoi** le regex a raté.

In [ ]:
# Construire le tableau des FN regex avec contexte
fn_rows = []
for r in llm_results:
    if r["error"]:
        continue
    R = set(r["regex_pks"])
    for raw in r["llm_pairs_raw"]:
        pk = raw["pair_key_norm"]
        if pk in R or not pk:
            continue
        # FN regex
        fn_rows.append({
            "jurisdiction": r["jurisdiction"],
            "id":           r["id"],
            "pair_key":     pk,
            "code_name":    raw["code_name"],
            "article_number": raw["article_number"],
            "verbatim":     raw["verbatim"][:150],
        })

df_fn = pd.DataFrame(fn_rows)
print(f"Total FN regex (LLM a trouvé, regex raté) : {len(df_fn)}")

if len(df_fn) > 0:
    # Top codes manqués
    print("\n▸ Top 15 codes les plus manqués par le regex")
    code_freq = df_fn["code_name"].value_counts().head(15)
    print(code_freq.to_string())

    print("\n▸ 20 exemples aléatoires de FN regex")
    print(df_fn.sample(min(20, len(df_fn)), random_state=SEED)[["jurisdiction", "code_name", "article_number", "verbatim"]].to_string(index=False))

In [ ]:
# Catégorisation automatique heuristique des FN
def categorize_fn(row) -> str:
    vb = (row["verbatim"] or "").lower()
    num = row["article_number"]
    # Heuristique simple — à raffiner en inspection manuelle
    if re.match(r".*c\.\s*(civ|pen|com|tra|proc)", vb):
        return "abréviation code (C. civ / C. pen etc.)"
    if "cpc" in vb or "c.p.c" in vb or "cpp" in vb:
        return "acronyme (CPC, CPP…)"
    if re.search(r"\barticles?\s+\d", vb) and "code" not in vb:
        return "article sans code explicite"
    if re.match(r"[LRDAE]\.?\s*\d", num.strip(), re.I) is None and re.match(r"\d", num.strip()) is None:
        return "numérotation atypique"
    return "autre"

if len(df_fn) > 0:
    df_fn["category"] = df_fn.apply(categorize_fn, axis=1)
    print("▸ Distribution des catégories de FN regex")
    print(df_fn["category"].value_counts().to_string())

---
## 9. Export des résultats + fiche de calibration

In [ ]:
from datetime import datetime

ts = datetime.now().strftime("%Y%m%d-%H%M")

# 1. CSV par arrêt
csv_path = RESULTS_DIR / f"regex_validation_per_record_{ts}.csv"
df.to_csv(csv_path, index=False)
print(f"✓ {csv_path}")

# 2. CSV FN regex
if len(df_fn) > 0:
    fn_csv = RESULTS_DIR / f"regex_validation_FN_{ts}.csv"
    df_fn.to_csv(fn_csv, index=False)
    print(f"✓ {fn_csv}")

# 3. JSON détaillé (un record complet par arrêt avec regex_pks, llm_pks, verbatims)
details_path = RESULTS_DIR / f"regex_validation_details_{ts}.json"
with open(details_path, "w", encoding="utf-8") as f:
    json.dump(llm_results, f, ensure_ascii=False, indent=2)
print(f"✓ {details_path}")

In [ ]:
# 4. Fiche de calibration markdown
md_path = RESULTS_DIR / f"regex_validation_fiche_{ts}.md"

lines = [
    f"# Fiche de calibration — regex d'extraction d'articles",
    "",
    f"**Date** : {datetime.now().isoformat(timespec='minutes')}  ",
    f"**Modèle LLM de référence** : `{MODEL_ID}`  ",
    f"**Température** : {TEMPERATURE}  ",
    f"**Échantillon** : {len(df)} arrêts — 50 CC + 25 CA + 25 TJ (seed={SEED})  ",
    "",
    "## Métriques globales (pooled)",
    "",
    f"| Métrique | Valeur |",
    f"|---|---:|",
    f"| Arrêts LLM valides | {len(df_ok)} / {len(df)} |",
    f"| Total pair_keys regex | {int(df_ok['n_regex'].sum())} |",
    f"| Total pair_keys LLM | {int(df_ok['n_llm'].sum())} |",
    f"| TP (accord) | {int(TP)} |",
    f"| FP (regex seul) | {int(FP)} |",
    f"| FN (regex raté) | {int(FN)} |",
    f"| **Précision regex** | **{P:.3f}** |",
    f"| **Recall regex** | **{R:.3f}** |",
    f"| **F1** | **{F1:.3f}** |",
    "",
    "## Par juridiction",
    "",
    by_jur.to_markdown(),
]

if len(df_fn) > 0:
    lines += [
        "",
        "## Distribution des catégories de FN regex",
        "",
        df_fn["category"].value_counts().to_markdown(),
        "",
        "## Top 15 codes manqués par le regex",
        "",
        df_fn["code_name"].value_counts().head(15).to_markdown(),
    ]

md_path.write_text("\n".join(lines), encoding="utf-8")
print(f"✓ {md_path}")

# Affichage
print("\n" + "═"*70)
print("RÉSUMÉ")
print("═"*70)
print(f"Précision regex vs LLM : {P:.3f}")
print(f"Recall regex vs LLM    : {R:.3f}")
print(f"F1                     : {F1:.3f}")
print(f"Arrêts où regex = LLM  : {(df['n_fp_regex']+df['n_fn_regex']==0).sum()}/{len(df)}")

---
## 10. Arrêt du serveur vLLM (optionnel)

In [ ]:
# À décommenter à la fin de la session pour libérer la VRAM
# import os, signal
# if VLLM_PID.exists():
#     pid = int(VLLM_PID.read_text())
#     os.killpg(os.getpgid(pid), signal.SIGTERM)
#     VLLM_PID.unlink()
#     print(f"✓ Serveur vLLM (PID={pid}) arrêté")